# CS-781 Capstone — Phase 5: Legal Statute Identification (LSI)

This notebook fine-tunes **your own Phase-1 language models** (A, B, C, D) to predict **all statutes** (out of 100 statute ids) that apply to a legal document — a multi-label task scored by macro-F1 over statutes — and writes one submission zip per experiment with a **descriptive file name**, e.g. `phase5_S1_B-GQA-RoPE_Adapter_LabelAttn_4x512_dev0.6512.zip`, all collected in `cs781_phase5/submissions/`.

## Before you run it
1. **Settings (right panel):** Accelerator **GPU T4 x2**, Internet **on**.
2. **Add-ons → Secrets:** turn on the secret **`HF`** (your Hugging Face token). The data is downloaded from `Exploration-Lab/CS-781-Capstone/Phase_5_LSI`.
3. **Add Input:** your Phase-1 dataset (the same one as in Phases 2–4). Section 6 prints which file is used for A, B, C and D.
4. *Save Version → Save & Run All.* To continue later, attach this notebook's previous output too: finished experiments are loaded, not retrained.

## The approach
- **Long documents:** each document is cut into chunks of 512 tokens; up to 4 chunks are read (2,048 tokens: the whole document for most cases, median ~450 words). Longer documents keep 4 chunks **spread evenly** over the text, always including the first and the last, because statutes can be cited anywhere.
- **Label attention head:** every statute has its own learned query that attends over all tokens of all chunks of the document, so each statute looks for its own evidence. One sigmoid per statute (multi-label).
- **Loss:** binary cross-entropy per statute; positives of rarer statutes weighted up (sqrt(negatives/positives), at most 10).
- **Thresholds:** one threshold for all statutes or one per statute — the notebook picks whichever scores better on a held-out half of dev, then tunes it on the whole dev set. A document with nothing above threshold gets its single most likely statute.
- Predictions contain only valid statute ids, no duplicates, one object per test document — checked before every zip is written.

| Submission | Model | Technique |
|---|---|---|
| S1_B_adapter | B (GQA/RoPE) | **bottleneck adapters** |
| S2_D_lorma | D (GPT, 400M tokens) | **LoRMA** (compulsory), on all linear layers, rank 32 |
| S3_A4_kd | A (first 4 layers) | **knowledge distillation** from S1 |
| S4_C_bitfit | C (encoder) | **BitFit** (+ token-embedding fix) |

Extra experiments for the analysis are defined in Section 2 (pooling head vs label attention, first chunk only, D with adapters); add their names to `RUN_EXPERIMENTS` to run them.

## Rules this follows
Only your Phase-1 models are used as starting points; no other pretrained model, weights or API. Training uses only the provided LSI data (dev only for model selection and thresholds).

## 1. Setup and Hugging Face login

In [1]:
import os
import re
import sys
import json
import math
import time
import glob
import gc
import pickle
import copy
import random
import shutil
import zipfile
import hashlib
import warnings
from collections import Counter, defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=FutureWarning)

HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF")
except Exception as e:
    HF_TOKEN = os.environ.get("HF_TOKEN")
    print("Kaggle secret 'HF' not available (" + type(e).__name__ + "); using the HF_TOKEN environment variable if set.")
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    try:
        from huggingface_hub import login
        login(token=HF_TOKEN, add_to_git_credential=False)
        print("Hugging Face login successful.")
    except Exception as e:
        print("Hugging Face login failed:", repr(e))

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
print("PyTorch:", torch.__version__, "| device:", DEVICE)
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))
if DEVICE.type == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
else:
    print("WARNING: no GPU. Use Kaggle 'GPU T4 x2' for real runs.")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hugging Face login successful.
PyTorch: 2.11.0+cu128 | device: cuda
GPU 0: Tesla T4
GPU 1: Tesla T4


## 2. Configuration and experiments

Each experiment is one line: which Phase-1 model, which technique, which head. `RUN_EXPERIMENTS` chooses what runs.

In [2]:
SEED = 42
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
PHASE5_DIR = "Phase_5_LSI"

INPUT_ROOT = os.environ.get("CS781_INPUT_ROOT", "/kaggle/input")
WORK_ROOT = os.environ.get("CS781_P5_WORK_ROOT",
                           "/kaggle/working/cs781_phase5" if os.path.isdir("/kaggle/working") else os.path.abspath("cs781_phase5"))
LOCAL_DATA_DIR = os.environ.get("CS781_P5_DATA_DIR", "")
SUBMISSION_DIR = os.path.join(WORK_ROOT, "submissions")
CACHE_DIR = os.path.join(WORK_ROOT, "cache")
for _d in (WORK_ROOT, SUBMISSION_DIR, CACHE_DIR):
    os.makedirs(_d, exist_ok=True)

PHASE1_MODELS = {
    "A": dict(architecture="A", pick="other"),
    "B": dict(architecture="B", pick="best"),
    "C": dict(architecture="C", pick="best"),
    "D": dict(architecture="A", pick="best"),
}
MODEL_TAGS = {"A": "A-GPT220M", "B": "B-GQA-RoPE", "C": "C-EncDec", "D": "D-GPT400M"}

DEFAULTS = dict(
    backbone="B", n_layers=None,
    technique="adapter",
    head="label_attention", attention_dim=256,
    chunk_tokens=512, max_chunks=4,
    chunks_per_batch=16,
    eval_chunks_per_batch=48,
    epochs=3, evals_per_epoch=1, patience=2,
    lr_backbone=3e-5, lr_peft=5e-4, lr_head=1e-3,
    weight_decay=0.01, warmup_frac=0.06, grad_clip=1.0,
    dropout=0.1, head_dropout=0.2,
    lora_r=16, lora_alpha=32, lora_scaling="standard", peft_targets="qv", lorma_variant="plus",
    adapter_dim=64, partial_top_layers=4,
    embedding_state=False, pooled_state_norm=False,
    pos_weight=True,
    teacher=None, kd_temperature=2.0, kd_weight=0.5,
    seed=SEED,
)

EXPERIMENTS = {
    "S1_B_adapter":    dict(backbone="B", technique="adapter"),
    "S2_D_lorma":      dict(backbone="D", technique="lorma", peft_targets="qv", lora_r=32, lora_alpha=64, epochs=1),
    "S2x_D_lorma_all": dict(backbone="D", technique="lorma", peft_targets="all", lora_r=32, lora_alpha=64),
    "S3_A4_kd":        dict(backbone="A", n_layers=4, technique="full", lr_backbone=5e-5,
                            teacher="S1_B_adapter"),
    "S4_C_bitfit":     dict(backbone="C", technique="bitfit", embedding_state=True, pooled_state_norm=True),
    "X1_B_adapter_pool":   dict(backbone="B", technique="adapter", head="pool"),
    "X2_B_adapter_1chunk": dict(backbone="B", technique="adapter", max_chunks=1),
    "X3_D_adapter":        dict(backbone="D", technique="adapter"),
}
RUN_EXPERIMENTS = ["S2_D_lorma"]
if os.environ.get("CS781_P5_RUN"):
    RUN_EXPERIMENTS = [x for x in os.environ["CS781_P5_RUN"].split(",") if x]

SMOKE_TEST = os.environ.get("CS781_P5_SMOKE", "0") == "1"
if SMOKE_TEST:
    DEFAULTS.update(chunk_tokens=48, chunks_per_batch=6, eval_chunks_per_batch=12, epochs=2, lora_r=4, adapter_dim=8,
                    attention_dim=16)
    for _cfg in EXPERIMENTS.values():
        if _cfg.get("n_layers"):
            _cfg["n_layers"] = 1
        if _cfg.get("lora_r"):
            _cfg["lora_r"], _cfg["lora_alpha"] = 4, 8
    print("*** SMOKE TEST MODE: tiny settings, results are meaningless ***")

def experiment_config(name):
    cfg = dict(DEFAULTS)
    cfg.update(EXPERIMENTS[name])
    cfg["name"] = name
    return cfg

def submission_name(cfg, dev_f1):
    """e.g. phase5_S1_B-GQA-RoPE_Adapter_LabelAttn_4x512_dev0.6512.zip"""
    technique = {"lorma": "LoRMA", "lora": "LoRA", "adapter": "Adapter", "bitfit": "BitFit", "full": "FullFT",
                 "partial": "PartialFT", "frozen": "Frozen"}[cfg["technique"]]
    if cfg["technique"] in ("lorma", "lora"):
        technique += f"-{cfg['peft_targets']}-r{cfg['lora_r']}"
    if cfg["teacher"]:
        technique = f"KD-from-{cfg['teacher'].split('_')[0]}"
    model = MODEL_TAGS[cfg["backbone"]] + (f"-first{cfg['n_layers']}layers" if cfg["n_layers"] else "")
    head = "LabelAttn" if cfg["head"] == "label_attention" else "MeanMaxPool"
    return (f"phase5_{cfg['name'].split('_')[0]}_{model}_{technique}_{head}_{cfg['max_chunks']}x{cfg['chunk_tokens']}"
            f"_dev{dev_f1:.4f}.zip")

def restore_previous_outputs():
    """Copy a previous run of this notebook (attached with "Add Input") into the working folder, so finished
    experiments are re-used instead of trained again."""
    if not os.path.isdir(INPUT_ROOT):
        return
    copied = 0
    for dirpath, dirnames, _ in os.walk(INPUT_ROOT):
        if os.path.basename(dirpath) == "cs781_phase5":
            for sub, _, files in os.walk(dirpath):
                for name in files:
                    src = os.path.join(sub, name)
                    dst = os.path.join(WORK_ROOT, os.path.relpath(src, dirpath))
                    if not os.path.exists(dst):
                        os.makedirs(os.path.dirname(dst), exist_ok=True)
                        shutil.copy2(src, dst)
                        copied += 1
            dirnames[:] = []
        elif dirpath[len(INPUT_ROOT):].count(os.sep) >= 6:
            dirnames[:] = []
    if copied:
        print(f"Restored {copied} file(s) from a previous Phase-5 run.")

restore_previous_outputs()
print("Working folder:", WORK_ROOT)
print("Experiments to run:", RUN_EXPERIMENTS)
for _name in RUN_EXPERIMENTS:
    print(f"  {_name:22s} -> {submission_name(experiment_config(_name), 0.0).replace('_dev0.0000.zip', '_dev<score>.zip')}")

Restored 38 file(s) from a previous Phase-5 run.
Working folder: /kaggle/working/cs781_phase5
Experiments to run: ['S2_D_lorma']
  S2_D_lorma             -> phase5_S2_D-GPT400M_LoRMA-qv-r32_LabelAttn_4x512_dev<score>.zip


## 3. Load the LSI data

In [3]:
def load_json_file(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def write_json(obj, path):
    with open(path + ".tmp", "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)
    os.replace(path + ".tmp", path)

_HF_FILES = None

def phase5_file(name):
    """Local path of a Phase-5 file such as "train.json" (local folder, attached dataset, or Hugging Face download)."""
    global _HF_FILES
    candidates = []
    if LOCAL_DATA_DIR:
        candidates += sorted(glob.glob(os.path.join(LOCAL_DATA_DIR, "**", name), recursive=True))
    if os.path.isdir(INPUT_ROOT):
        for depth in range(0, 4):
            candidates += sorted(glob.glob(os.path.join(INPUT_ROOT, "*", *(["*"] * depth), PHASE5_DIR, "**", name), recursive=True))
    for c in candidates:
        if os.path.isfile(c):
            return c
    from huggingface_hub import hf_hub_download, list_repo_files
    if _HF_FILES is None:
        _HF_FILES = list_repo_files(HF_DATASET_REPO, repo_type="dataset", token=HF_TOKEN)
    matches = sorted((f for f in _HF_FILES if f.startswith(PHASE5_DIR + "/") and f.split("/")[-1] == name), key=len)
    if not matches:
        raise FileNotFoundError(f"{name} not found in {HF_DATASET_REPO}/{PHASE5_DIR}")
    print(f"downloading {matches[0]} ...", flush=True)
    return hf_hub_download(repo_id=HF_DATASET_REPO, filename=matches[0], repo_type="dataset", token=HF_TOKEN)

def read_cases(name, split):
    """-> list of dicts {id (exactly as in the file), text, n_words, labels (list of statute ids) or None}."""
    data = load_json_file(phase5_file(name))
    if isinstance(data, dict):
        data = [dict(v, id=k) if isinstance(v, dict) else {"id": k, "text": v} for k, v in data.items()]
    out = []
    for i, d in enumerate(tqdm(data, desc=f"reading {name}", leave=False)):
        text = " ".join(str(d.get("text") or "").split())
        case = {"id": d.get("id", i), "text": text, "n_words": len(text.split(" ")) if text else 0, "labels": None}
        if split != "test":
            labels = d.get("labels")
            if labels is None:
                raise ValueError(f"{name}: case {case['id']} has no labels")
            if not isinstance(labels, (list, tuple)):
                labels = [labels]
            case["labels"] = sorted({int(x) for x in labels})
        out.append(case)
    del data
    gc.collect()
    return out

CASES = {"train": read_cases("train.json", "train"), "dev": read_cases("dev.json", "dev"), "test": read_cases("test.json", "test")}
_test_ids = [json.dumps(c["id"]) for c in CASES["test"]]
assert len(set(_test_ids)) == len(_test_ids), "duplicate test ids"

STATUTE_IDS = sorted({l for s in ("train", "dev") for c in CASES[s] for l in c["labels"]})
STATUTE_INDEX = {sid: k for k, sid in enumerate(STATUTE_IDS)}
NUM_LABELS = len(STATUTE_IDS)

def label_matrix(split):
    y = np.zeros((len(CASES[split]), NUM_LABELS), dtype=np.float32)
    for i, c in enumerate(CASES[split]):
        for l in c["labels"]:
            y[i, STATUTE_INDEX[l]] = 1.0
    return y

Y = {"train": label_matrix("train"), "dev": label_matrix("dev")}

_upper = {s: float(np.mean([sum(ch.isupper() for ch in c["text"][:3000]) / max(1, len(c["text"][:3000]))
                            for c in CASES[s][:500]])) for s in CASES}
LOWERCASE = min(_upper.values()) < 0.005 < max(_upper.values())
if LOWERCASE:
    for _s in CASES:
        for _c in CASES[_s]:
            _c["text"] = _c["text"].lower()
print("Upper-case share per split:", {k: round(v, 4) for k, v in _upper.items()},
      "-> text lower-cased" if LOWERCASE else "-> casing kept (all splits alike)")

LENGTH_BUCKETS = [(0, 500, "<500 words"), (500, 1500, "500-1.5k"), (1500, 5000, "1.5k-5k"), (5000, 10 ** 12, ">5k")]
print(f"\n{'split':6s} {'cases':>7s} {'median words':>13s} {'max words':>10s} {'statutes/case':>14s}")
for _s, _cs in CASES.items():
    _w = [c["n_words"] for c in _cs]
    _l = f"{np.mean([len(c['labels']) for c in _cs]):.2f}" if _s != "test" else "-"
    print(f"{_s:6s} {len(_cs):7,} {int(np.median(_w)):13,} {max(_w):10,} {_l:>14s}")
_freq = Y["train"].sum(0)
print(f"\n{NUM_LABELS} statutes (ids {STATUTE_IDS[0]}..{STATUTE_IDS[-1]}); training documents per statute: "
      f"min {int(_freq.min())}, median {int(np.median(_freq))}, max {int(_freq.max())}")
print("Example (train, first 300 characters):", CASES["train"][0]["text"][:300])

downloading Phase_5_LSI/Train/train.json ...


Phase_5_LSI/Train/train.json: reconstructing file:   0%|          |  0.00B /  299MB            

Phase_5_LSI/Train/train.json: downloading bytes:           |  0.00B            

reading train.json:   0%|          | 0/42750 [00:00<?, ?it/s]

downloading Phase_5_LSI/Train/dev.json ...


Phase_5_LSI/Train/dev.json: reconstructing file:   0%|          |  0.00B / 79.4MB            

... (3 outputs left out)

Phase_5_LSI/Test/test.json: reconstructing file:   0%|          |  0.00B / 98.1MB            

Phase_5_LSI/Test/test.json: downloading bytes:           |  0.00B            

reading test.json:   0%|          | 0/13019 [00:00<?, ?it/s]

Upper-case share per split: {'train': 0.0987, 'dev': 0.0973, 'test': 0.1016} -> casing kept (all splits alike)

split    cases  median words  max words  statutes/case
train   42,750           430     86,858           3.70
dev     10,181           543     69,230           3.92
test    13,019           532    134,848              -

100 statutes (ids 0..99); training documents per statute: min 272, median 679, max 11730
Example (train, first 300 characters): (a), <SECTION> of the <ACT>. The gist of theprosecution case relevant <ENTITY> the purpose of this proceedingmay be stated thus: With the growth of industry, commerceand trade in and around the city of Mumbai which generatessubstantial quantity of wealth, there has been increase oforganised activiti


## 4. Phase-1 tokenizer and document chunking

In [4]:
SPECIAL_TOKENS = ["<pad>", "<unk>", "<s>", "</s>"]
PAD_ID = 0

class BPETokenizer:
    """Loads the byte-level BPE tokenizer saved by a Phase-1 notebook (tokenizer.json) and encodes text with the
    same merge rules. Only loading and encoding are needed here; the tokenizer is NOT retrained."""

    def __init__(self, directory):
        with open(os.path.join(directory, "tokenizer.json"), "r", encoding="utf-8") as f:
            data = json.load(f)
        self.token_to_id = {str(k): int(v) for k, v in data["token_to_id"].items()}
        self.merge_ranks = {tuple(pair): i for i, pair in enumerate(data["merges"])}
        self.unk_id = self.token_to_id["<unk>"]
        self.eow_id = self.token_to_id["</w>"]
        self.vocab_size = len(self.token_to_id)
        self._cache = {}

    def _merge(self, text):
        tokens = [str(b) for b in text.encode("utf-8")]
        while len(tokens) > 1:
            best_rank, best_index = None, -1
            for i in range(len(tokens) - 1):
                rank = self.merge_ranks.get((tokens[i], tokens[i + 1]))
                if rank is not None and (best_rank is None or rank < best_rank):
                    best_rank, best_index = rank, i
            if best_index < 0:
                break
            tokens[best_index:best_index + 2] = [tokens[best_index] + tokens[best_index + 1]]
        return [self.token_to_id.get(t, self.unk_id) for t in tokens]

    def encode_piece(self, piece):
        """Token ids of one piece of text WITHOUT the end-of-word token."""
        ids = self._cache.get(piece)
        if ids is None:
            ids = []
            for k in range(0, len(piece), 200):
                ids.extend(self._merge(piece[k:k + 200]))
            if len(self._cache) < 2_000_000:
                self._cache[piece] = ids
        return ids

def document_ids(text, tokenizer, max_tokens):
    """Every word BPE-encoded and followed by </w>. Stops after max_tokens tokens (a cap for extreme documents)."""
    ids = []
    for word in text.split():
        ids.extend(tokenizer.encode_piece(word))
        ids.append(tokenizer.eow_id)
        if len(ids) >= max_tokens:
            break
    return ids[:max_tokens] or [tokenizer.eow_id]

def select_chunks(ids, chunk_tokens, max_chunks, min_tokens=16):
    """Cut a document into consecutive chunks of chunk_tokens. A document with more than max_chunks chunks keeps
    max_chunks of them spread EVENLY over the document (always the first and the last one), because statutes can
    be cited anywhere. A very short last piece is dropped unless it is the only one. -> list of arrays."""
    ids = np.asarray(ids, dtype=np.int32)
    pieces = [ids[i:i + chunk_tokens] for i in range(0, len(ids), chunk_tokens)]
    if len(pieces) > 1 and len(pieces[-1]) < min_tokens:
        pieces = pieces[:-1]
    if len(pieces) > max_chunks:
        keep = np.unique(np.round(np.linspace(0, len(pieces) - 1, max_chunks)).astype(int))
        pieces = [pieces[i] for i in keep]
    return pieces

## 5. Metrics and threshold selection

In [5]:
THRESHOLD_GRID = np.round(np.arange(0.05, 0.951, 0.01), 2)

def ensure_one(pred, prob):
    """A document with no statute above its threshold gets its single most likely statute (every case cites one)."""
    pred = pred.copy()
    empty = ~pred.any(1)
    pred[np.where(empty)[0], prob[empty].argmax(1)] = True
    return pred

def multilabel_scores(gold, pred):
    """gold, pred: [N, L] 0/1 -> macro-F1 / P / R (mean over ALL statute columns, like sklearn average="macro"),
    micro-F1, subset accuracy (exact set match) and per-statute F1."""
    gold, pred = gold.astype(bool), pred.astype(bool)
    tp = (gold & pred).sum(0).astype(np.float64)
    fp = (~gold & pred).sum(0).astype(np.float64)
    fn = (gold & ~pred).sum(0).astype(np.float64)
    precision = np.divide(tp, tp + fp, out=np.zeros_like(tp), where=(tp + fp) > 0)
    recall = np.divide(tp, tp + fn, out=np.zeros_like(tp), where=(tp + fn) > 0)
    f1 = np.divide(2 * tp, 2 * tp + fp + fn, out=np.zeros_like(tp), where=(2 * tp + fp + fn) > 0)
    micro = 2 * tp.sum() / max(1.0, 2 * tp.sum() + fp.sum() + fn.sum())
    return {"macro_f1": float(f1.mean()), "macro_precision": float(precision.mean()), "macro_recall": float(recall.mean()),
            "micro_f1": float(micro), "subset_accuracy": float((gold == pred).all(1).mean()),
            "predicted_per_case": float(pred.sum(1).mean()), "per_label_f1": f1.tolist(),
            "per_label_support": gold.sum(0).astype(int).tolist()}

def _per_label_f1_curve(gold, prob):
    """F1 of every statute at every grid threshold -> [len(grid), L]."""
    gold = gold.astype(bool)
    out = np.zeros((len(THRESHOLD_GRID), gold.shape[1]))
    for i, t in enumerate(THRESHOLD_GRID):
        p = prob >= t
        tp = (gold & p).sum(0)
        denom = 2 * tp + (~gold & p).sum(0) + (gold & ~p).sum(0)
        out[i] = np.divide(2 * tp, denom, out=np.zeros(gold.shape[1]), where=denom > 0)
    return out

def tune_thresholds(gold, prob, strategy):
    """strategy "global": one threshold for all statutes (max macro-F1); "per_label": each statute its own threshold
    (max that statute's F1). -> array [L] of thresholds."""
    curve = _per_label_f1_curve(gold, prob)
    if strategy == "global":
        return np.full(gold.shape[1], THRESHOLD_GRID[int(curve.mean(1).argmax())])
    return THRESHOLD_GRID[curve.argmax(0)]

def predict_with(prob, thresholds):
    return ensure_one(prob >= thresholds[None, :], prob)

def choose_thresholds(gold, prob, seed=0):
    """Thresholds are tuned on dev, so a dev score with them is optimistic. To pick the strategy honestly, dev is
    split in two halves: thresholds tuned on one half are scored on the other (and vice versa). The strategy with the
    better held-out macro-F1 is then tuned on the whole dev set. -> (thresholds, strategy, report)."""
    rng = np.random.RandomState(seed)
    half = rng.permutation(len(gold)) < len(gold) // 2
    report = {}
    for strategy in ("global", "per_label"):
        scores = []
        for a, b in ((half, ~half), (~half, half)):
            t = tune_thresholds(gold[a], prob[a], strategy)
            scores.append(multilabel_scores(gold[b], predict_with(prob[b], t))["macro_f1"])
        report[strategy] = float(np.mean(scores))
    strategy = max(report, key=report.get)
    return tune_thresholds(gold, prob, strategy), strategy, report

def quick_dev_f1(gold, prob):
    """Model selection during training: macro-F1 at the best single global threshold."""
    t = tune_thresholds(gold, prob, "global")
    return multilabel_scores(gold, predict_with(prob, t))["macro_f1"], float(t[0])

def scores_by_length(cases, gold, pred):
    out = {}
    for lo, hi, lab in LENGTH_BUCKETS:
        idx = [i for i, c in enumerate(cases) if lo <= c["n_words"] < hi]
        if idx:
            s = multilabel_scores(gold[idx], pred[idx])
            out[lab] = {"cases": len(idx), "macro_f1": s["macro_f1"], "micro_f1": s["micro_f1"]}
    return out

## 6. Phase-1 models

In [6]:
ARCH_OF_TYPE = {"decoder-only": "A", "decoder-only-gqa": "B", "encoder-decoder": "C", "efficient-local-mqa": "D-alibi"}
ARCH_DESCRIPTION = {"A": "GPT decoder", "B": "GQA/RoPE decoder", "C": "encoder-decoder", "D-alibi": "local MQA/ALiBi decoder"}

def torch_load(path):
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(path, map_location="cpu")

def discover_phase1_checkpoints(root):
    """Finds the models written by the Phase-1 notebooks:
      <...>/<ARCH>/legal-model-final/{model.pt, config.json, tokenizer.json}       (normal case), or
      <...>/<ARCH>/best_model.pt + <...>/common/bpe_tokenizer_<ARCH>/tokenizer.json  (run stopped before the final save)."""
    found = []
    if not os.path.isdir(root):
        return found
    for dirpath, dirnames, filenames in os.walk(root):
        if dirpath[len(root):].count(os.sep) >= 12:
            dirnames[:] = []
        try:
            if os.path.basename(dirpath) == "legal-model-final" and "model.pt" in filenames and "tokenizer.json" in filenames:
                config = load_json_file(os.path.join(dirpath, "config.json"))
                history_path = os.path.join(dirpath, "training_history.json")
                model_file, tokenizer_dir = os.path.join(dirpath, "model.pt"), dirpath
                if os.path.exists(history_path):
                    eval_loss = float(load_json_file(history_path).get("best_eval_loss", "inf"))
                else:
                    eval_loss = float(torch_load(model_file).get("best_eval_loss", "inf"))
            elif "best_model.pt" in filenames and "legal-model-final" not in dirnames:
                common = os.path.join(os.path.dirname(dirpath), "common")
                tokenizer_dirs = [d for d in sorted(glob.glob(os.path.join(common, "bpe_tokenizer_" + os.path.basename(dirpath) + "*")))
                                  if os.path.exists(os.path.join(d, "tokenizer.json"))]
                if not tokenizer_dirs:
                    continue
                saved = torch_load(os.path.join(dirpath, "best_model.pt"))
                config, eval_loss = saved["config"], float(saved.get("eval_loss", "inf"))
                del saved
                model_file, tokenizer_dir = os.path.join(dirpath, "best_model.pt"), tokenizer_dirs[-1]
            else:
                continue
            arch = ARCH_OF_TYPE.get(config.get("type")) or str(config.get("architecture", "?"))[:1]
        except Exception as e:
            print("Skipping", dirpath, "(" + type(e).__name__ + ")")
            continue
        found.append({"arch": arch, "dir": dirpath, "model_file": model_file, "tokenizer_dir": tokenizer_dir, "config": config,
                      "eval_loss": eval_loss, "size_mb": os.path.getsize(model_file) / 1e6})
    return found

PHASE1_FOUND = discover_phase1_checkpoints(INPUT_ROOT)
PHASE1 = {}
print(f"Phase-1 checkpoints found under {INPUT_ROOT}: {len(PHASE1_FOUND)}")
for c in sorted(PHASE1_FOUND, key=lambda c: (c["arch"], c["eval_loss"])):
    print(f"  {ARCH_DESCRIPTION.get(c['arch'], c['arch']):24s} val loss {c['eval_loss']:.4f} | {c['size_mb']:.0f} MB | {c['dir']}")
_chosen = {}
for slot, spec in sorted(PHASE1_MODELS.items(), key=lambda kv: kv[1]["pick"] == "other"):
    arch, choice = spec["architecture"], spec["pick"]
    candidates = [c for c in PHASE1_FOUND if c["arch"] == arch]
    if choice == "other":
        taken = [_chosen[a]["eval_loss"] for a in _chosen if _chosen[a]["arch"] == arch]
        candidates = [c for c in candidates if all(abs(c["eval_loss"] - t) > 1e-6 for t in taken)]
        pick = max(candidates, key=lambda c: c["eval_loss"]) if candidates else None
    else:
        if choice not in ("best", "auto"):
            candidates = [c for c in candidates if choice.lower() in c["dir"].lower()]
        pick = min(candidates, key=lambda c: c["eval_loss"]) if candidates else None
    if pick is not None:
        PHASE1[slot] = _chosen[slot] = dict(pick, slot=slot)
print("Your Phase-1 models:")
for slot in PHASE1_MODELS:
    if slot in PHASE1:
        e = PHASE1[slot]
        print(f"  {slot}: {ARCH_DESCRIPTION.get(e['arch'], e['arch'])}, validation loss {e['eval_loss']:.4f} | {e['dir']}")
    else:
        print(f"  {slot}: NOT FOUND - attach the output of that Phase-1 notebook with 'Add Input'")

TOKENIZERS = {}

def tokenizer_for(slot):
    path = PHASE1[slot]["tokenizer_dir"]
    if path not in TOKENIZERS:
        TOKENIZERS[path] = BPETokenizer(path)
    return TOKENIZERS[path]

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x):
        rms = torch.rsqrt(x.float().pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return (x * rms.to(x.dtype)) * self.weight

class AttentionA(nn.Module):
    def __init__(self, d, n_heads, dropout):
        super().__init__()
        self.n_heads, self.head_dim, self.dropout = n_heads, d // n_heads, dropout
        self.qkv = nn.Linear(d, 3 * d, bias=True)
        self.out = nn.Linear(d, d, bias=True)

    def forward(self, x):
        b, t, c = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        q, k, v = [z.view(b, t, self.n_heads, self.head_dim).transpose(1, 2) for z in (q, k, v)]
        y = F.scaled_dot_product_attention(q, k, v, dropout_p=self.dropout if self.training else 0.0, is_causal=True)
        return self.out(y.transpose(1, 2).contiguous().view(b, t, c))

class BlockA(nn.Module):
    def __init__(self, d, n_heads, d_ff, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(d)
        self.attn = AttentionA(d, n_heads, dropout)
        self.ln2 = nn.LayerNorm(d)
        self.fc1 = nn.Linear(d, d_ff)
        self.fc2 = nn.Linear(d_ff, d)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = x + self.dropout(self.attn(self.ln1(x)))
        return x + self.dropout(self.fc2(F.gelu(self.fc1(self.ln2(x)))))

class BackboneA(nn.Module):
    bidirectional = False

    def __init__(self, cfg, vocab_size, n_layers, dropout):
        super().__init__()
        d = cfg["d_model"]
        self.d_model, self.max_tokens = d, cfg["context_length"]
        self.token_embedding = nn.Embedding(vocab_size, d)
        self.position_embedding = nn.Embedding(cfg["context_length"], d)
        self.blocks = nn.ModuleList([BlockA(d, cfg["n_head"], cfg["d_ff"], dropout) for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(d)

    def layers(self):
        return self.blocks

    def forward(self, input_ids, pad_mask=None):
        positions = torch.arange(input_ids.size(1), device=input_ids.device)
        x = self.token_embedding(input_ids) + self.position_embedding(positions)[None]
        states = []
        for block in self.blocks:
            x = block(x)
            states.append(self.ln_f(x))
        return states

def rotate_half(x):
    return torch.stack((-x[..., 1::2], x[..., ::2]), dim=-1).flatten(-2)

class AttentionB(nn.Module):
    def __init__(self, d, n_heads, n_kv_heads, max_len, dropout):
        super().__init__()
        self.q_heads, self.kv_heads, self.head_dim, self.dropout = n_heads, n_kv_heads, d // n_heads, dropout
        self.q_proj = nn.Linear(d, d, bias=False)
        self.k_proj = nn.Linear(d, n_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(d, n_kv_heads * self.head_dim, bias=False)
        self.o_proj = nn.Linear(d, d, bias=False)
        inv_freq = 1.0 / (10000.0 ** (torch.arange(0, self.head_dim, 2).float() / self.head_dim))
        freqs = torch.outer(torch.arange(max_len).float(), inv_freq)
        self.register_buffer("cos", torch.cos(freqs).repeat_interleave(2, dim=-1), persistent=False)
        self.register_buffer("sin", torch.sin(freqs).repeat_interleave(2, dim=-1), persistent=False)

    def forward(self, x):
        b, t, c = x.shape
        q = self.q_proj(x).view(b, t, self.q_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(b, t, self.kv_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(b, t, self.kv_heads, self.head_dim).transpose(1, 2)
        cos, sin = self.cos[:t].to(q.dtype)[None, None], self.sin[:t].to(q.dtype)[None, None]
        q = q * cos + rotate_half(q) * sin
        k = k * cos + rotate_half(k) * sin
        repeat = self.q_heads // self.kv_heads
        if repeat > 1:
            k, v = k.repeat_interleave(repeat, dim=1), v.repeat_interleave(repeat, dim=1)
        y = F.scaled_dot_product_attention(q, k, v, dropout_p=self.dropout if self.training else 0.0, is_causal=True)
        return self.o_proj(y.transpose(1, 2).contiguous().view(b, t, c))

class SwiGLU(nn.Module):
    def __init__(self, d, d_ff):
        super().__init__()
        self.gate = nn.Linear(d, d_ff, bias=False)
        self.up = nn.Linear(d, d_ff, bias=False)
        self.down = nn.Linear(d_ff, d, bias=False)

    def forward(self, x):
        return self.down(F.silu(self.gate(x)) * self.up(x))

class BlockB(nn.Module):
    def __init__(self, d, n_heads, n_kv_heads, d_ff, max_len, dropout):
        super().__init__()
        self.norm1 = RMSNorm(d)
        self.attn = AttentionB(d, n_heads, n_kv_heads, max_len, dropout)
        self.norm2 = RMSNorm(d)
        self.mlp = SwiGLU(d, d_ff)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = x + self.dropout(self.attn(self.norm1(x)))
        return x + self.dropout(self.mlp(self.norm2(x)))

class BackboneB(nn.Module):
    bidirectional = False

    def __init__(self, cfg, vocab_size, n_layers, dropout):
        super().__init__()
        d = cfg["d_model"]
        self.d_model, self.max_tokens = d, cfg["context_length"]
        self.token_embedding = nn.Embedding(vocab_size, d)
        self.blocks = nn.ModuleList([BlockB(d, cfg["n_head"], cfg["n_kv_head"], cfg["d_ff"], cfg["context_length"], dropout)
                                     for _ in range(n_layers)])
        self.norm_f = RMSNorm(d)

    def layers(self):
        return self.blocks

    def forward(self, input_ids, pad_mask=None):
        x = self.token_embedding(input_ids)
        states = []
        for block in self.blocks:
            x = block(x)
            states.append(self.norm_f(x))
        return states

class AttentionC(nn.Module):
    def __init__(self, d, n_heads, dropout):
        super().__init__()
        self.n_heads, self.head_dim, self.dropout = n_heads, d // n_heads, dropout
        self.q_proj = nn.Linear(d, d)
        self.k_proj = nn.Linear(d, d)
        self.v_proj = nn.Linear(d, d)
        self.o_proj = nn.Linear(d, d)

    def forward(self, x, attn_mask=None):
        b, t, c = x.shape
        q, k, v = [proj(x).view(b, t, self.n_heads, self.head_dim).transpose(1, 2)
                   for proj in (self.q_proj, self.k_proj, self.v_proj)]
        y = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask, dropout_p=self.dropout if self.training else 0.0)
        return self.o_proj(y.transpose(1, 2).contiguous().view(b, t, c))

class EncoderBlockC(nn.Module):
    def __init__(self, d, n_heads, d_ff, dropout):
        super().__init__()
        self.self_attn = AttentionC(d, n_heads, dropout)
        self.ln1 = nn.LayerNorm(d)
        self.ff1 = nn.Linear(d, d_ff)
        self.ff2 = nn.Linear(d_ff, d)
        self.ln2 = nn.LayerNorm(d)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, attn_mask=None):
        x = self.ln1(x + self.dropout(self.self_attn(x, attn_mask)))
        return self.ln2(x + self.dropout(self.ff2(F.gelu(self.ff1(x)))))

class BackboneC(nn.Module):
    bidirectional = True

    def __init__(self, cfg, vocab_size, n_layers, dropout):
        super().__init__()
        d = cfg["d_model"]
        self.d_model, self.max_tokens = d, cfg["encoder_length"]
        self.token_embedding = nn.Embedding(vocab_size, d)
        self.encoder = nn.ModuleList([EncoderBlockC(d, cfg["n_head"], cfg["d_ff"], dropout) for _ in range(n_layers)])
        pe = torch.zeros(self.max_tokens, d)
        position = torch.arange(0, self.max_tokens).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d, 2).float() * (-math.log(10000.0) / d))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe, persistent=False)

    def layers(self):
        return self.encoder

    embedding_state = False

    def forward(self, input_ids, pad_mask=None):
        x = self.token_embedding(input_ids)
        states = []
        if self.embedding_state:
            scale = self.token_embedding.weight.detach().float().pow(2).mean().sqrt().clamp_min(1e-12)
            states.append(F.layer_norm(x / scale.to(x.dtype), (self.d_model,)))
        x = x + self.pe[:x.size(1)].to(x.dtype)[None]
        attn_mask = None if pad_mask is None else pad_mask[:, None, None, :]
        for block in self.encoder:
            x = block(x, attn_mask)
            states.append(x)
        return states

class AttentionD(nn.Module):
    def __init__(self, d, n_heads, dropout):
        super().__init__()
        self.n_heads, self.head_dim, self.dropout = n_heads, d // n_heads, dropout
        self.q_proj = nn.Linear(d, d, bias=False)
        self.k_proj = nn.Linear(d, self.head_dim, bias=False)
        self.v_proj = nn.Linear(d, self.head_dim, bias=False)
        self.o_proj = nn.Linear(d, d, bias=False)

    def forward(self, x, attn_bias):
        b, t, c = x.shape
        q = self.q_proj(x).view(b, t, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(b, t, 1, self.head_dim).transpose(1, 2).expand(-1, self.n_heads, -1, -1)
        v = self.v_proj(x).view(b, t, 1, self.head_dim).transpose(1, 2).expand(-1, self.n_heads, -1, -1)
        y = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_bias.to(q.dtype),
                                           dropout_p=self.dropout if self.training else 0.0)
        return self.o_proj(y.transpose(1, 2).contiguous().view(b, t, c))

class BlockD(nn.Module):
    def __init__(self, d, n_heads, d_ff, dropout):
        super().__init__()
        self.norm1 = RMSNorm(d)
        self.attn = AttentionD(d, n_heads, dropout)
        self.norm2 = RMSNorm(d)
        self.fc1 = nn.Linear(d, d_ff)
        self.fc2 = nn.Linear(d_ff, d)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, attn_bias):
        x = x + self.dropout(self.attn(self.norm1(x), attn_bias))
        return x + self.dropout(self.fc2(F.gelu(self.fc1(self.norm2(x)))))

class BackboneD(nn.Module):
    bidirectional = False

    def __init__(self, cfg, vocab_size, n_layers, dropout):
        super().__init__()
        d = cfg["d_model"]
        self.d_model, self.max_tokens, self.window = d, cfg["context_length"], cfg["local_window"]
        self.token_embedding = nn.Embedding(vocab_size, d)
        self.blocks = nn.ModuleList([BlockD(d, cfg["n_head"], cfg["d_ff"], dropout) for _ in range(n_layers)])
        self.norm_f = RMSNorm(d)
        n = cfg["n_head"]
        self.register_buffer("slopes", 2.0 ** (-(8.0 * torch.arange(n) / n)), persistent=False)

    def layers(self):
        return self.blocks

    def attention_bias(self, t, device):
        pos = torch.arange(t, device=device, dtype=torch.float32)
        distance = pos[:, None] - pos[None, :]
        blocked = (distance < 0) | (distance > self.window - 1)
        bias = -self.slopes.to(device)[:, None, None] * distance[None]
        return bias.masked_fill(blocked[None], float("-inf"))[None]

    def forward(self, input_ids, pad_mask=None):
        x = self.token_embedding(input_ids)
        attn_bias = self.attention_bias(input_ids.size(1), input_ids.device)
        states = []
        for block in self.blocks:
            x = block(x, attn_bias)
            states.append(self.norm_f(x))
        return states

BACKBONE_CLASSES = {"A": BackboneA, "B": BackboneB, "C": BackboneC, "D-alibi": BackboneD}

@torch.no_grad()
def encoder_token_check(backbone, length=256):
    """Does the Phase-1 encoder output depend on the input tokens? Two different random token sequences are encoded;
    if the last-layer states at the same positions are almost identical, the encoder only encodes positions."""
    was_training = backbone.training
    backbone.eval()
    g = torch.Generator().manual_seed(0)
    vocab = backbone.token_embedding.weight.shape[0]
    length = min(length, backbone.max_tokens)
    ids = torch.randint(4, vocab, (2, length), generator=g)
    flag, backbone.embedding_state = backbone.embedding_state, False
    last = backbone(ids)[-1].float()
    backbone.embedding_state = flag
    same_position = F.cosine_similarity(last[0], last[1], dim=-1).mean().item()
    other_position = F.cosine_similarity(last[0], last[0].roll(1, 0), dim=-1).mean().item()
    emb_rms = backbone.token_embedding.weight.float().pow(2).mean().sqrt().item()
    pe_rms = backbone.pe[:length].float().pow(2).mean().sqrt().item()
    print(f"Encoder check: token-embedding RMS {emb_rms:.4f} vs position-encoding RMS {pe_rms:.4f} | last layer, "
          f"cosine similarity of two DIFFERENT inputs at the same position {same_position:.3f} "
          f"(neighbouring positions of one input {other_position:.3f}; close to 1.0 = the output ignores the tokens)")
    backbone.train(was_training)

def load_backbone(slot, n_layers=None, dropout=0.1, verbose=True):
    """Build the architecture and load its Phase-1 weights (optionally only the first n layers)."""
    entry = PHASE1[slot]
    arch = entry["arch"]
    cfg = entry["config"]
    state = torch_load(entry["model_file"])["model"]
    total_layers = cfg["encoder_layers"] if arch == "C" else cfg["n_layer"]
    n_layers = total_layers if n_layers is None else min(int(n_layers), total_layers)
    vocab_size = state["token_embedding.weight"].shape[0]
    backbone = BACKBONE_CLASSES[arch](cfg, vocab_size, n_layers, dropout)
    wanted = set(backbone.state_dict().keys())
    missing = wanted - set(state.keys())
    if missing:
        raise RuntimeError(f"Phase-1 checkpoint for {slot} lacks weights: {sorted(missing)[:5]} ...")
    backbone.load_state_dict({k: v for k, v in state.items() if k in wanted}, strict=True)
    if verbose and arch == "C":
        encoder_token_check(backbone)
    if verbose:
        unused = len(state) - len(wanted)
        print(f"Loaded Phase-1 model {slot} ({ARCH_DESCRIPTION.get(arch, arch)}): {n_layers}/{total_layers} layers, "
              f"{sum(p.numel() for p in backbone.parameters()) / 1e6:.2f}M parameters"
              + (f" ({unused} weight tensors not needed, e.g. removed layers or the decoder)" if unused else ""))
    return backbone

Phase-1 checkpoints found under /kaggle/input: 4
  GPT decoder              val loss 1.3086 | 258 MB | /kaggle/input/datasets/prathamk26/cs781-phase1/untitled folder/results-D/cs781_phase1/A_GPT_MHA_LearnedPos/legal-model-final
  GPT decoder              val loss 1.7840 | 258 MB | /kaggle/input/datasets/prathamk26/cs781-phase1/untitled folder/results-A/cs781_phase1/A_GPT_MHA_LearnedPos/legal-model-final
  GQA/RoPE decoder         val loss 1.6521 | 264 MB | /kaggle/input/datasets/prathamk26/cs781-phase1/untitled folder/results-B/cs781_phase1/B_GQA_RoPE_RMSNorm_SwiGLU/legal-model-final
  encoder-decoder          val loss 3.0858 | 422 MB | /kaggle/input/datasets/prathamk26/cs781-phase1/untitled folder/results-C/cs781_phase1/C_EncoderDecoder/legal-model-final
Your Phase-1 models:
  A: GPT decoder, validation loss 1.7840 | /kaggle/input/datasets/prathamk26/cs781-phase1/untitled folder/results-A/cs781_phase1/A_GPT_MHA_LearnedPos/legal-model-final
  B: GQA/RoPE decoder, validation loss 1.6521

## 7. Fine-tuning and adaptation techniques

| Technique | What is trained |
|---|---|
| `full` | every weight |
| `partial` | the top layers only |
| `bitfit` | biases and normalisation gains |
| `lorma` | low-rank multiplicative update (I + (α/r)·BA)·W0 (LoRMA+) |
| `lora` | low-rank additive update W0 + (α/r)·BA |
| `adapter` | bottleneck adapters after attention and feed-forward (+ norms) |
| `frozen` | head only |

In [7]:
class LoRALinear(nn.Module):
    """LoRA (Hu et al.): additive low-rank update,  W = W0 + (alpha / r) * B A.   B starts at zero.
    With scaling="rs" it is rsLoRA:  W = W0 + (alpha / sqrt(r)) * B A."""

    def __init__(self, base, r, alpha, rows=None, scaling="standard"):
        super().__init__()
        self.base, self.scale = base, (alpha / math.sqrt(r) if scaling == "rs" else alpha / r)
        self.rows = rows or [(0, base.out_features)]
        self.lora_A = nn.ParameterList([nn.Parameter(torch.randn(r, base.in_features) / math.sqrt(base.in_features))
                                        for _ in self.rows])
        self.lora_B = nn.ParameterList([nn.Parameter(torch.zeros(e - s, r)) for s, e in self.rows])

    def forward(self, x):
        y = self.base(x)
        parts, cursor = [], 0
        for (s, e), A, B in zip(self.rows, self.lora_A, self.lora_B):
            if s > cursor:
                parts.append(y[..., cursor:s])
            parts.append(y[..., s:e] + self.scale * F.linear(F.linear(x, A.to(x.dtype)), B.to(x.dtype)))
            cursor = e
        if cursor < y.size(-1):
            parts.append(y[..., cursor:])
        return parts[0] if len(parts) == 1 else torch.cat(parts, dim=-1)

_PERM_INDEX = {}

class LoRMALinear(nn.Module):
    """LoRMA (Low-Rank Multiplicative Adaptation): the pretrained weight is MULTIPLIED by a learned matrix.

      "plus" (LoRMA+):   W = (I + (alpha / r) * B A) W0     B starts at zero, so the multiplier starts as the identity.
      "pi"   (LoRMA-pi): W = P(B A) W0, where P shifts row i of (B A) cyclically by i positions, which makes a
                         low-rank product full rank. B's first column is ones and A[0, 0] = 1 (rest of A zero), so
                         P(B A) starts as the identity.
    Because the multiplier is on the left of W0, it acts on the layer's OUTPUT: h = W0 x, y = M h + bias.
    """

    def __init__(self, base, r, alpha, rows=None, variant="plus"):
        super().__init__()
        self.base, self.scale, self.variant = base, alpha / r, variant
        self.rows = rows or [(0, base.out_features)]
        self.lorma_A, self.lorma_B = nn.ParameterList(), nn.ParameterList()
        for s, e in self.rows:
            m = e - s
            if variant == "plus":
                A, B = torch.randn(r, m) / math.sqrt(m), torch.zeros(m, r)
            elif variant == "pi":
                A, B = torch.zeros(r, m), torch.randn(m, r) * 0.02
                A[0, 0] = 1.0
                B[:, 0] = 1.0
            else:
                raise ValueError("lorma_variant must be 'plus' or 'pi'")
            self.lorma_A.append(nn.Parameter(A))
            self.lorma_B.append(nn.Parameter(B))

    @staticmethod
    def _permuted(BA):
        m = BA.size(0)
        key = (m, BA.device)
        if key not in _PERM_INDEX:
            i = torch.arange(m, device=BA.device)
            _PERM_INDEX[key] = (i[None, :] - i[:, None]) % m
        return torch.gather(BA, 1, _PERM_INDEX[key])

    def forward(self, x):
        h = F.linear(x, self.base.weight)
        parts, cursor = [], 0
        for (s, e), A, B in zip(self.rows, self.lorma_A, self.lorma_B):
            if s > cursor:
                parts.append(h[..., cursor:s])
            hs = h[..., s:e]
            if self.variant == "plus":
                upd = self.scale * F.linear(F.linear(hs.float(), A.float()), B.float())
                parts.append((hs.float() + upd).to(hs.dtype))
            else:
                parts.append(F.linear(hs, self._permuted(B @ A).to(hs.dtype)))
            cursor = e
        if cursor < h.size(-1):
            parts.append(h[..., cursor:])
        y = parts[0] if len(parts) == 1 else torch.cat(parts, dim=-1)
        return y if self.base.bias is None else y + self.base.bias

class AdapterLinear(nn.Module):
    """Houlsby-style bottleneck adapter placed on a sub-layer's output, before the residual connection:
    y = h + up(GELU(down(h))),  h = base(x).   `up` starts at zero, so the adapter starts as the identity."""

    def __init__(self, base, bottleneck):
        super().__init__()
        self.base = base
        self.adapter_down = nn.Linear(base.out_features, bottleneck)
        self.adapter_up = nn.Linear(bottleneck, base.out_features)
        nn.init.normal_(self.adapter_down.weight, std=0.02)
        nn.init.zeros_(self.adapter_down.bias)
        nn.init.zeros_(self.adapter_up.weight)
        nn.init.zeros_(self.adapter_up.bias)

    def forward(self, x):
        h = self.base(x)
        return h + self.adapter_up(F.gelu(self.adapter_down(h)))

class IA3Linear(nn.Module):
    """IA3 (Liu et al., "(IA)^3"): learned vectors that RESCALE activations, initialised to ones (= no change).
    mode "out": rescale the layer's output (used for attention keys and values; `rows` limits it to part of a fused
    matrix).  mode "in": rescale the layer's input (used for the feed-forward hidden activations, before fc2)."""

    def __init__(self, base, mode, rows=None):
        super().__init__()
        self.base, self.mode = base, mode
        if mode == "in":
            self.ia3_scale = nn.Parameter(torch.ones(base.in_features))
            self.rows = None
        else:
            self.rows = rows or [(0, base.out_features)]
            index = torch.cat([torch.arange(s, e) for s, e in self.rows])
            self.register_buffer("row_index", index, persistent=False)
            self.ia3_scale = nn.Parameter(torch.ones(len(index)))

    def forward(self, x):
        if self.mode == "in":
            return self.base(x * self.ia3_scale.to(x.dtype))
        y = self.base(x)
        full = torch.ones(y.size(-1), device=y.device, dtype=y.dtype).scatter(0, self.row_index, self.ia3_scale.to(y.dtype))
        return y * full

QV_TARGETS = {"A": ["attn.qkv"], "B": ["attn.q_proj", "attn.v_proj"], "C": ["self_attn.q_proj", "self_attn.v_proj"],
              "D-alibi": ["attn.q_proj", "attn.v_proj"]}
ALL_TARGETS = {"A": ["attn.qkv", "attn.out", "fc1", "fc2"],
               "B": ["attn.q_proj", "attn.k_proj", "attn.v_proj", "attn.o_proj", "mlp.gate", "mlp.up", "mlp.down"],
               "C": ["self_attn.q_proj", "self_attn.k_proj", "self_attn.v_proj", "self_attn.o_proj", "ff1", "ff2"],
               "D-alibi": ["attn.q_proj", "attn.k_proj", "attn.v_proj", "attn.o_proj", "fc1", "fc2"]}
SUBLAYER_OUTPUTS = {"A": ["attn.out", "fc2"], "B": ["attn.o_proj", "mlp.down"], "C": ["self_attn.o_proj", "ff2"],
                    "D-alibi": ["attn.o_proj", "fc2"]}
IA3_TARGETS = {"A": [("attn.qkv", "out"), ("fc2", "in")], "B": [("attn.k_proj", "out"), ("attn.v_proj", "out"), ("mlp.down", "in")],
               "C": [("self_attn.k_proj", "out"), ("self_attn.v_proj", "out"), ("ff2", "in")],
               "D-alibi": [("attn.k_proj", "out"), ("attn.v_proj", "out"), ("fc2", "in")]}
NEW_WEIGHT_MARKERS = ("lora_", "lorma_", "adapter_", "ia3_")

def _get(module, path):
    for part in path.split("."):
        module = getattr(module, part)
    return module

def _set(module, path, value):
    parts = path.split(".")
    for part in parts[:-1]:
        module = getattr(module, part)
    setattr(module, parts[-1], value)

def apply_technique(backbone, arch, cfg):
    """Freeze the pretrained model, then make trainable exactly what the chosen technique allows."""
    technique = cfg["technique"]
    for p in backbone.parameters():
        p.requires_grad = False
    blocks = backbone.layers()
    norm_types = (nn.LayerNorm, RMSNorm)
    if technique == "full":
        for p in backbone.parameters():
            p.requires_grad = True
    elif technique == "partial":
        for block in list(blocks)[-int(cfg["partial_top_layers"]):]:
            for p in block.parameters():
                p.requires_grad = True
        for name in ("ln_f", "norm_f"):
            if hasattr(backbone, name):
                for p in getattr(backbone, name).parameters():
                    p.requires_grad = True
    elif technique == "bitfit":
        for name, p in backbone.named_parameters():
            if name.endswith(".bias"):
                p.requires_grad = True
        for module in backbone.modules():
            if isinstance(module, norm_types):
                for p in module.parameters():
                    p.requires_grad = True
    elif technique in ("lora", "lorma"):
        targets = (QV_TARGETS if cfg["peft_targets"] == "qv" else ALL_TARGETS)[arch]
        for block in blocks:
            for path in targets:
                base = _get(block, path)
                rows = None
                if arch == "A" and path == "attn.qkv" and cfg["peft_targets"] == "qv":
                    d = base.in_features
                    rows = [(0, d), (2 * d, 3 * d)]
                if technique == "lora":
                    wrapped = LoRALinear(base, cfg["lora_r"], cfg["lora_alpha"], rows, cfg.get("lora_scaling", "standard"))
                else:
                    wrapped = LoRMALinear(base, cfg["lora_r"], cfg["lora_alpha"], rows, cfg["lorma_variant"])
                _set(block, path, wrapped)
    elif technique == "ia3":
        for block in blocks:
            for path, mode in IA3_TARGETS[arch]:
                base = _get(block, path)
                rows = None
                if arch == "A" and path == "attn.qkv":
                    d = base.in_features
                    rows = [(d, 2 * d), (2 * d, 3 * d)]
                _set(block, path, IA3Linear(base, mode, rows))
    elif technique == "adapter":
        for block in blocks:
            for path in SUBLAYER_OUTPUTS[arch]:
                _set(block, path, AdapterLinear(_get(block, path), cfg["adapter_dim"]))
        for module in backbone.modules():
            if isinstance(module, norm_types):
                for p in module.parameters():
                    p.requires_grad = True
    elif technique != "frozen":
        raise ValueError("Unknown technique: " + str(technique))
    for name, p in backbone.named_parameters():
        if any(marker in name for marker in NEW_WEIGHT_MARKERS):
            p.requires_grad = True
    return backbone

## 8. Model: chunks → language model → label attention → one sigmoid per statute

In [8]:
class LSIClassifier(nn.Module):
    """Phase-1 language model + multi-label statute head.

    1. Every chunk of the document (up to max_chunks x chunk_tokens tokens) is read by the language model.
    2. Token vectors: a learned weighted average of ALL layers ("scalar mix"), then LayerNorm.
    3. head="label_attention": each statute k has a learned query q_k; it attends over ALL tokens of ALL chunks of
       the document (a_k = softmax(q_k . tanh(W h_t))), giving one document vector per statute, and its score is
       w_k . sum_t a_k,t h_t + b_k. Different statutes can look at different places in the document.
       head="pool": the mean and the max of the chunk vectors, then one linear layer for all statutes.
    4. One independent sigmoid per statute (multi-label): a document can get any number of statutes.
    """

    def __init__(self, backbone, cfg):
        super().__init__()
        self.backbone, self.head_type = backbone, cfg["head"]
        d = backbone.d_model
        has_embedding_state = bool(getattr(backbone, "embedding_state", False))
        self.mix = nn.Parameter(torch.zeros(len(backbone.layers()) + int(has_embedding_state)))
        self.state_norm = bool(cfg.get("pooled_state_norm", False))
        if self.state_norm and has_embedding_state:
            with torch.no_grad():
                self.mix[0] = math.log(max(1, len(backbone.layers())))
        self.norm = nn.LayerNorm(d)
        self.drop = nn.Dropout(cfg["head_dropout"])
        if self.head_type == "label_attention":
            self.att_proj = nn.Linear(d, cfg["attention_dim"])
            self.label_queries = nn.Parameter(torch.randn(NUM_LABELS, cfg["attention_dim"]) * 0.02)
            self.label_weight = nn.Parameter(torch.randn(NUM_LABELS, d) * 0.02)
            self.label_bias = nn.Parameter(torch.zeros(NUM_LABELS))
        elif self.head_type == "pool":
            self.out = nn.Linear(2 * d, NUM_LABELS)
        else:
            raise ValueError("unknown head " + str(cfg["head"]))

    def token_states(self, input_ids, pad_mask):
        """[N, T] ids of N chunks -> [N, T, d] float32 token vectors (scalar mix of all layers, LayerNorm)."""
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            states = self.backbone(input_ids, pad_mask)
        weights = torch.softmax(self.mix, dim=0)
        assert len(states) == len(weights), (len(states), len(weights))
        if self.state_norm:
            h = sum(w * F.layer_norm(s.float(), (s.size(-1),)) for w, s in zip(weights, states))
        else:
            h = sum(w * s.float() for w, s in zip(weights, states))
        return self.norm(h)

    def forward(self, input_ids, pad_mask, chunk_doc, n_docs):
        """input_ids/pad_mask [N, T] for N chunks; chunk_doc [N] = which document (0..n_docs-1) each chunk belongs to.
        -> statute scores [n_docs, L] (logits)."""
        h = self.drop(self.token_states(input_ids, pad_mask))
        if self.head_type == "pool":
            m = pad_mask[:, :, None].float()
            chunk_vec = (h * m).sum(1) / m.sum(1).clamp(min=1.0)
            d = chunk_vec.size(-1)
            mean = torch.zeros(n_docs, d, device=h.device).index_add_(0, chunk_doc, chunk_vec)
            count = torch.zeros(n_docs, device=h.device).index_add_(0, chunk_doc, torch.ones_like(chunk_doc, dtype=torch.float))
            mean = mean / count.clamp(min=1.0)[:, None]
            mx = torch.full((n_docs, d), -1e4, device=h.device)
            mx = mx.scatter_reduce(0, chunk_doc[:, None].expand(-1, d), chunk_vec, reduce="amax", include_self=True)
            return self.out(self.drop(torch.cat([mean, mx], dim=-1)))
        n, t, d = h.shape
        slot = torch.zeros(n, dtype=torch.long, device=h.device)
        counts = torch.bincount(chunk_doc, minlength=n_docs)
        starts = torch.cumsum(counts, 0) - counts
        slot = torch.arange(n, device=h.device) - starts[chunk_doc]
        k = int(counts.max().item())
        doc_h = h.new_zeros(n_docs, k, t, d)
        doc_m = torch.zeros(n_docs, k, t, dtype=torch.bool, device=h.device)
        doc_h[chunk_doc, slot] = h
        doc_m[chunk_doc, slot] = pad_mask
        doc_h, doc_m = doc_h.view(n_docs, k * t, d), doc_m.view(n_docs, k * t)
        u = torch.tanh(self.att_proj(doc_h))
        scores = torch.einsum("bsa,la->bls", u, self.label_queries)
        scores = scores.masked_fill(~doc_m[:, None, :], float("-inf"))
        att = torch.softmax(scores, dim=-1)
        z = torch.einsum("bls,bsd->bld", att, doc_h)
        return (z * self.label_weight[None]).sum(-1) + self.label_bias[None]

def multilabel_loss(logits, targets, cfg, pos_weight=None, teacher=None):
    """Binary cross-entropy per statute (positives up-weighted for rare statutes), plus knowledge distillation:
    the student matches the teacher's softened per-statute probabilities sigmoid(z / T)."""
    hard = F.binary_cross_entropy_with_logits(logits, targets, pos_weight=pos_weight)
    if teacher is None:
        return hard
    T, w = float(cfg["kd_temperature"]), float(cfg["kd_weight"])
    soft = F.binary_cross_entropy_with_logits(logits / T, torch.sigmoid(teacher / T)) * T * T
    return (1 - w) * hard + w * soft

## 9. Training and prediction

Batches are whole documents (up to 16 chunks per step). One dev evaluation per epoch; the best one is kept, and an interrupted run resumes from its last evaluation.

In [9]:
CHUNKS = {}
MAX_DOC_TOKENS = 200_000

def document_chunks(slot, chunk_tokens, max_chunks):
    """{split: [list of np.int32 chunks of every case]} for the tokenizer of Phase-1 model `slot`."""
    tokenizer_dir = PHASE1[slot]["tokenizer_dir"]
    sizes = "-".join(str(len(CASES[s])) for s in ("train", "dev", "test"))
    key = hashlib.md5(f"{tokenizer_dir}|{chunk_tokens}|{max_chunks}|{sizes}|{LOWERCASE}|{MAX_DOC_TOKENS}".encode()).hexdigest()[:12]
    if key in CHUNKS:
        return CHUNKS[key]
    path = os.path.join(CACHE_DIR, f"chunks_{max_chunks}x{chunk_tokens}_{key}.pkl")
    if os.path.exists(path):
        with open(path, "rb") as f:
            CHUNKS[key] = pickle.load(f)
        print(f"tokenised documents for model {slot}: loaded from {path}")
        return CHUNKS[key]
    tokenizer = tokenizer_for(slot)
    out = {}
    for split in ("train", "dev", "test"):
        out[split] = [select_chunks(document_ids(c["text"], tokenizer, MAX_DOC_TOKENS), chunk_tokens, max_chunks)
                      for c in tqdm(CASES[split], desc=f"tokenising {split} for model {slot}")]
    with open(path + ".tmp", "wb") as f:
        pickle.dump(out, f, protocol=4)
    os.replace(path + ".tmp", path)
    CHUNKS[key] = out
    n = [len(x) for x in out["train"]]
    print(f"tokenised documents for model {slot}: chunks per training case: mean {np.mean(n):.2f}; "
          f"cases using all {max_chunks} chunks: {sum(k == max_chunks for k in n):,} / {len(n):,}")
    return out

def doc_batches(docs, chunks, budget, order):
    """Group documents (in the given order) into batches holding at most `budget` chunks in total."""
    batches, cur, used = [], [], 0
    for i in order:
        k = len(chunks[i])
        if cur and used + k > budget:
            batches.append(cur)
            cur, used = [], 0
        cur.append(i)
        used += k
    if cur:
        batches.append(cur)
    return batches

def make_batch(chunks, doc_idx):
    """-> ids [N, T], mask [N, T], chunk_doc [N] (chunks of one document are consecutive)."""
    seqs, owner = [], []
    for j, i in enumerate(doc_idx):
        for c in chunks[i]:
            seqs.append(c)
            owner.append(j)
    width = max(len(s) for s in seqs)
    ids = np.full((len(seqs), width), PAD_ID, dtype=np.int64)
    mask = np.zeros((len(seqs), width), dtype=bool)
    for r, s in enumerate(seqs):
        ids[r, :len(s)], mask[r, :len(s)] = s, True
    return (torch.from_numpy(ids).to(DEVICE), torch.from_numpy(mask).to(DEVICE),
            torch.tensor(owner, dtype=torch.long, device=DEVICE))

@torch.no_grad()
def predict_logits(model, chunks, budget, desc="predict"):
    """Statute scores [N_docs, L] (float32) for a list of chunked documents, in their original order."""
    model.eval()
    order = sorted(range(len(chunks)), key=lambda i: (len(chunks[i]), max(len(c) for c in chunks[i])))
    out = np.zeros((len(chunks), NUM_LABELS), dtype=np.float32)
    for batch in tqdm(doc_batches(None, chunks, budget, order), desc=desc, leave=False):
        ids, mask, owner = make_batch(chunks, batch)
        out[batch] = model(ids, mask, owner, len(batch)).float().cpu().numpy()
    return out

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -30, 30)))

def exp_dir(name):
    path = os.path.join(WORK_ROOT, name)
    os.makedirs(path, exist_ok=True)
    return path

def build_model(cfg, verbose=True):
    backbone = load_backbone(cfg["backbone"], cfg["n_layers"], cfg["dropout"], verbose=verbose)
    if cfg["chunk_tokens"] > backbone.max_tokens:
        raise ValueError(f"chunk_tokens {cfg['chunk_tokens']} > the model's limit {backbone.max_tokens}")
    apply_technique(backbone, PHASE1[cfg["backbone"]]["arch"], cfg)
    if cfg.get("embedding_state"):
        if not hasattr(backbone, "embedding_state"):
            raise ValueError("embedding_state is only implemented for the encoder (model C)")
        backbone.embedding_state = True
    return LSIClassifier(backbone, cfg).to(DEVICE)

def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    backbone = sum(p.numel() for p in model.backbone.parameters())
    backbone_trainable = sum(p.numel() for p in model.backbone.parameters() if p.requires_grad)
    return {"total": total, "trainable": trainable, "trainable_percent": 100.0 * trainable / max(1, total),
            "language_model": backbone, "language_model_trainable": backbone_trainable, "head": total - backbone}

def trainable_state(model):
    return {name: p.detach().cpu().clone() for name, p in model.named_parameters() if p.requires_grad}

def load_trainable(model, state):
    result = model.load_state_dict(state, strict=False)
    if result.unexpected_keys:
        raise RuntimeError("Saved weights do not fit this model: " + str(result.unexpected_keys[:5]))

def save_atomic(obj, path):
    torch.save(obj, path + ".tmp")
    os.replace(path + ".tmp", path)

def config_signature(cfg, extra=""):
    entry = PHASE1[cfg["backbone"]]
    payload = {"cfg": {k: v for k, v in cfg.items() if k != "name"}, "phase1": [entry["config"], entry["eval_loss"]],
               "data": [len(CASES[s]) for s in ("train", "dev", "test")], "labels": STATUTE_IDS,
               "lowercase": LOWERCASE, "extra": extra}
    return hashlib.md5(json.dumps(payload, sort_keys=True, default=str).encode()).hexdigest()

def optimizer_groups(model, cfg):
    backbone_lr = cfg["lr_peft"] if cfg["technique"] in ("bitfit", "adapter", "ia3") else cfg["lr_backbone"]
    buckets = defaultdict(list)
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if not name.startswith("backbone."):
            lr = cfg["lr_head"]
        elif any(marker in name for marker in NEW_WEIGHT_MARKERS):
            lr = cfg["lr_peft"]
        else:
            lr = backbone_lr
        decay = cfg["weight_decay"] if (p.ndim >= 2 and "label_queries" not in name) else 0.0
        buckets[(lr, decay)].append(p)
    return [{"params": params, "lr": lr, "base_lr": lr, "weight_decay": decay} for (lr, decay), params in buckets.items()]

def lr_factor(step, total_steps, warmup_frac):
    warmup = max(1, int(total_steps * warmup_frac))
    if step < warmup:
        return (step + 1) / warmup
    progress = min(1.0, (step - warmup) / max(1, total_steps - warmup))
    return 0.05 + 0.95 * 0.5 * (1.0 + math.cos(math.pi * progress))

def release(*objects):
    for obj in objects:
        del obj
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

def pos_weight_tensor(y):
    pos = y.sum(0)
    w = np.sqrt((len(y) - pos) / np.maximum(pos, 1.0))
    return torch.tensor(np.clip(w, 1.0, 10.0), dtype=torch.float32, device=DEVICE)

def train_model(cfg, signature, teacher_logits=None):
    """Train on train.json; evaluate on dev evals_per_epoch times per epoch; keep the evaluation with the best dev
    macro-F1 (at the best global threshold). Resumes from the last evaluation if the session was interrupted."""
    folder = exp_dir(cfg["name"])
    last_path, model_path = os.path.join(folder, "last.pt"), os.path.join(folder, "model.pt")
    torch.manual_seed(cfg["seed"])
    model = build_model(cfg)
    params = count_parameters(model)
    print(f"[{cfg['name']}] parameters: {params['total'] / 1e6:.2f}M total | trained: "
          f"{params['language_model_trainable'] / 1e6:.3f}M of the {params['language_model'] / 1e6:.2f}M language-model weights "
          f"+ {params['head'] / 1e6:.3f}M head ({params['trainable_percent']:.2f}% of all weights)")
    chunks = document_chunks(cfg["backbone"], cfg["chunk_tokens"], cfg["max_chunks"])
    train_chunks, dev_chunks = chunks["train"], chunks["dev"]
    y_train, y_dev = Y["train"], Y["dev"]
    epoch_batches = [doc_batches(None, train_chunks, cfg["chunks_per_batch"],
                                 np.random.RandomState(cfg["seed"] * 1000 + e).permutation(len(train_chunks)))
                     for e in range(cfg["epochs"])]
    epoch_start = np.cumsum([0] + [len(b) for b in epoch_batches])
    total_steps = int(epoch_start[-1])
    steps_per_epoch = total_steps / cfg["epochs"]
    optimizer = torch.optim.AdamW(optimizer_groups(model, cfg), betas=(0.9, 0.98))
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    pos_weight = pos_weight_tensor(y_train) if cfg["pos_weight"] else None

    state = {"step": 0, "history": [], "best_f1": -1.0, "best_step": 0, "bad_evals": 0,
             "seconds": 0.0, "peak_memory_mb": 0.0, "loss_sum": 0.0, "n_loss": 0}
    if os.path.exists(last_path):
        saved = torch_load(last_path)
        if saved.get("signature") == signature:
            load_trainable(model, saved["model"])
            optimizer.load_state_dict(saved["optimizer"])
            scaler.load_state_dict(saved["scaler"])
            state = saved["state"]
            print(f"[{cfg['name']}] resuming after step {state['step']:,} of {total_steps:,}")
        del saved
    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats()

    def evaluate_now(epoch_float):
        t_eval = time.time()
        prob = sigmoid(predict_logits(model, dev_chunks, cfg["eval_chunks_per_batch"], desc="dev evaluation"))
        f1, thr = quick_dev_f1(y_dev, prob)
        record = {"step": state["step"], "epoch": round(epoch_float, 2),
                  "train_loss": state["loss_sum"] / max(1, state["n_loss"]), "dev_macro_f1": f1, "dev_threshold": thr}
        state["loss_sum"], state["n_loss"] = 0.0, 0
        message = (f"[{cfg['name']}] epoch {epoch_float:4.2f} | train loss {record['train_loss']:.4f} | dev macro-F1 "
                   f"{f1:.4f} (one threshold {thr:.2f})")
        if f1 > state["best_f1"]:
            state.update(best_f1=f1, best_step=state["step"], best_epoch=record["epoch"], bad_evals=0)
            save_atomic({"model": trainable_state(model), "signature": signature, "step": state["step"]}, model_path)
            message += "  <- best"
        else:
            state["bad_evals"] += 1
        state["history"].append(record)
        if DEVICE.type == "cuda":
            state["peak_memory_mb"] = max(state["peak_memory_mb"], torch.cuda.max_memory_allocated() / 1e6)
        save_atomic({"model": trainable_state(model), "optimizer": optimizer.state_dict(), "scaler": scaler.state_dict(),
                     "state": state, "signature": signature}, last_path)
        print(message + f" | eval {time.time() - t_eval:.0f} s", flush=True)
        model.train()

    stopped_early = state["bad_evals"] >= cfg["patience"]
    while state["step"] < total_steps and not stopped_early:
        epoch = int(np.searchsorted(epoch_start, state["step"], side="right") - 1)
        first = state["step"] - int(epoch_start[epoch])
        batches = epoch_batches[epoch]
        n_batches = len(batches)
        eval_every = max(1, n_batches // max(1, cfg["evals_per_epoch"]))
        torch.manual_seed(cfg["seed"] * 1000 + epoch + first)
        model.train()
        t0 = time.time()
        bar = tqdm(range(first, n_batches), desc=f"{cfg['name']} epoch {epoch + 1}/{cfg['epochs']}", leave=False)
        for b in bar:
            idx = batches[b]
            for group in optimizer.param_groups:
                group["lr"] = group["base_lr"] * lr_factor(state["step"], total_steps, cfg["warmup_frac"])
            ids, mask, owner = make_batch(train_chunks, idx)
            logits = model(ids, mask, owner, len(idx)).float()
            teacher = torch.from_numpy(teacher_logits[idx]).to(DEVICE) if teacher_logits is not None else None
            loss = multilabel_loss(logits, torch.from_numpy(y_train[idx]).to(DEVICE), cfg, pos_weight, teacher)
            optimizer.zero_grad(set_to_none=True)
            if not torch.isfinite(loss):
                state["step"] += 1
                state["skipped"] = state.get("skipped", 0) + 1
                if state["skipped"] in (1, 10, 100) or state["skipped"] % 1000 == 0:
                    print(f"[{cfg['name']}] WARNING: non-finite loss, batch skipped ({state['skipped']} so far)", flush=True)
                if b != n_batches - 1:
                    continue
                state["step"] -= 1
            else:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], cfg["grad_clip"])
                scaler.step(optimizer)
                scaler.update()
                state["loss_sum"] += loss.item()
                state["n_loss"] += 1
            state["step"] += 1
            if b % 20 == 0:
                bar.set_postfix(loss=f"{state['loss_sum'] / max(1, state['n_loss']):.4f}")
            at_epoch_end = b == n_batches - 1
            if ((b + 1) % eval_every == 0 and (n_batches - (b + 1)) >= eval_every // 2) or at_epoch_end:
                state["seconds"] += time.time() - t0
                t0 = time.time()
                evaluate_now(epoch + (b + 1) / n_batches)
                if state["bad_evals"] >= cfg["patience"]:
                    stopped_early = True
                    break
        bar.close()
    if stopped_early:
        print(f"[{cfg['name']}] early stop: no improvement in {cfg['patience']} evaluations.")
    load_trainable(model, torch_load(model_path)["model"])
    if os.path.exists(last_path):
        os.remove(last_path)
    info = {"best_step": state["best_step"], "best_epoch": state.get("best_epoch", round(state["best_step"] / steps_per_epoch, 2)),
            "history": state["history"], "train_seconds": state["seconds"], "peak_memory_mb": state["peak_memory_mb"],
            "parameters": params, "train_cases": len(train_chunks)}
    return model, info

def load_model(cfg):
    model = build_model(cfg, verbose=False)
    load_trainable(model, torch_load(os.path.join(exp_dir(cfg["name"]), "model.pt"))["model"])
    return model

## 10. Run the experiments

For each experiment: train → dev scores and thresholds → test predictions → descriptive zip in `submissions/`. Finished experiments are loaded from disk. One failing experiment does not stop the others.

In [10]:
RESULTS = {}

def write_predictions(cfg, folder, pred, dev_f1):
    """prediction.json = [{"id": <case id exactly as in test.json>, "pred": [statute ids]}, ...]: one object per test
    case, valid integer ids only, no duplicates, at least one id each. Zipped under a descriptive name in the
    experiment folder AND in submissions/. Everything is checked before the file is written."""
    output = []
    for c, row in zip(CASES["test"], pred):
        ids = sorted({int(STATUTE_IDS[k]) for k in np.where(row)[0]})
        output.append({"id": c["id"], "pred": ids})
    assert len(output) == len(CASES["test"])
    prediction_path = os.path.join(folder, "prediction.json")
    with open(prediction_path, "w", encoding="utf-8") as f:
        json.dump(output, f, ensure_ascii=False)
    zip_name = submission_name(cfg, dev_f1)
    for old in glob.glob(os.path.join(folder, "phase5_*.zip")) + \
            glob.glob(os.path.join(SUBMISSION_DIR, f"phase5_{cfg['name'].split('_')[0]}_*.zip")):
        os.remove(old)
    zip_path = os.path.join(folder, zip_name)
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
        z.write(prediction_path, arcname="prediction.json")
    with zipfile.ZipFile(zip_path, "r") as z:
        assert z.namelist() == ["prediction.json"]
        check = json.loads(z.read("prediction.json").decode("utf-8"))
    valid = set(int(s) for s in STATUTE_IDS)
    assert [r["id"] for r in check] == [c["id"] for c in CASES["test"]], "test ids changed"
    for r in check:
        assert isinstance(r["pred"], list) and r["pred"], r
        assert all(isinstance(x, int) and x in valid for x in r["pred"]) and len(set(r["pred"])) == len(r["pred"]), r
    shutil.copy2(zip_path, os.path.join(SUBMISSION_DIR, zip_name))
    return os.path.join(SUBMISSION_DIR, zip_name)

def teacher_logits_for(cfg):
    """Teacher's statute scores for every training case (its own tokenizer), cached in the student's folder."""
    teacher_cfg = experiment_config(cfg["teacher"])
    path = os.path.join(exp_dir(cfg["name"]), f"teacher_logits_{RESULTS[cfg['teacher']]['signature'][:10]}.npy")
    if os.path.exists(path):
        return np.load(path)
    teacher = load_model(teacher_cfg)
    chunks = document_chunks(teacher_cfg["backbone"], teacher_cfg["chunk_tokens"], teacher_cfg["max_chunks"])["train"]
    logits = predict_logits(teacher, chunks, teacher_cfg["eval_chunks_per_batch"], desc=f"teacher {cfg['teacher']} scores")
    print(f"[{cfg['name']}] distilling from {cfg['teacher']} ({count_parameters(teacher)['total'] / 1e6:.1f}M parameters)")
    release(teacher)
    np.save(path, logits)
    return logits

def run_experiment(name):
    """Train (best evaluation on dev) -> thresholds on dev -> test predictions -> submission zip.
    Every finished part is loaded from disk."""
    if name in RESULTS:
        return RESULTS[name]
    cfg = experiment_config(name)
    if cfg["backbone"] not in PHASE1:
        print(f"[{name}] SKIPPED: no Phase-1 checkpoint for model {cfg['backbone']} is attached.")
        return None
    teacher_signature = ""
    if cfg["teacher"]:
        dep = run_experiment(cfg["teacher"])
        if dep is None:
            print(f"[{name}] SKIPPED: its teacher {cfg['teacher']} could not be run.")
            return None
        teacher_signature = dep["signature"]
    folder = exp_dir(name)
    signature = config_signature(cfg, teacher_signature)
    results_path = os.path.join(folder, "results.json")
    if os.path.exists(results_path):
        saved = load_json_file(results_path)
        if saved.get("signature") == signature and os.path.exists(saved.get("submission", "")):
            print(f"[{name}] already finished: dev macro-F1 {saved['dev']['macro_f1']:.4f} (loaded from disk)")
            RESULTS[name] = saved
            return saved

    model_path, info_path = os.path.join(folder, "model.pt"), os.path.join(folder, "train_info.json")
    trained = (os.path.exists(model_path) and os.path.exists(info_path) and not os.path.exists(os.path.join(folder, "last.pt"))
               and torch_load(model_path).get("signature") == signature)
    if trained:
        model = load_model(cfg)
        info = load_json_file(info_path)
        print(f"[{name}] model already trained; loading it.")
    else:
        teacher = teacher_logits_for(cfg) if cfg["teacher"] else None
        model, info = train_model(cfg, signature, teacher)
        write_json(info, info_path)
    chunks = document_chunks(cfg["backbone"], cfg["chunk_tokens"], cfg["max_chunks"])

    dev_prob = sigmoid(predict_logits(model, chunks["dev"], cfg["eval_chunks_per_batch"], desc="dev prediction"))
    thresholds, strategy, held_out = choose_thresholds(Y["dev"], dev_prob, cfg["seed"])
    dev_pred = predict_with(dev_prob, thresholds)
    dev = multilabel_scores(Y["dev"], dev_pred)
    dev.update(threshold_strategy=strategy, held_out_macro_f1=held_out, thresholds=thresholds.tolist(),
               by_length=scores_by_length(CASES["dev"], Y["dev"], dev_pred), best_epoch=info["best_epoch"],
               history=info["history"], parameters=info["parameters"], train_seconds=info["train_seconds"],
               peak_memory_mb=info["peak_memory_mb"])
    np.save(os.path.join(folder, "dev_probabilities.npy"), dev_prob.astype(np.float16))
    print(f"[{name}] DEV macro-F1 {dev['macro_f1']:.4f} with {strategy} thresholds (held-out estimate: "
          f"one threshold {held_out['global']:.4f}, per statute {held_out['per_label']:.4f}) | micro-F1 {dev['micro_f1']:.4f} | "
          f"exact-set accuracy {dev['subset_accuracy']:.4f} | statutes per case {dev['predicted_per_case']:.2f} "
          f"(gold {Y['dev'].sum(1).mean():.2f}) | best at epoch {info['best_epoch']}")

    t0 = time.time()
    test_prob = sigmoid(predict_logits(model, chunks["test"], cfg["eval_chunks_per_batch"], desc="test prediction"))
    inference_seconds = time.time() - t0
    test_pred = predict_with(test_prob, thresholds)
    np.save(os.path.join(folder, "test_probabilities.npy"), test_prob.astype(np.float16))
    zip_path = write_predictions(cfg, folder, test_pred, dev["macro_f1"])
    result = {"name": name, "signature": signature, "config": {k: v for k, v in cfg.items() if k != "name"},
              "phase1_checkpoint": PHASE1[cfg["backbone"]]["dir"], "dev": dev, "test_cases": len(test_pred),
              "test_statutes_per_case": float(test_pred.sum(1).mean()), "inference_seconds": inference_seconds,
              "cases_per_second": len(test_pred) / max(1e-9, inference_seconds), "submission": zip_path}
    write_json(result, results_path)
    print(f"[{name}] test: {len(test_pred):,} cases, {test_pred.sum(1).mean():.2f} statutes per case -> {zip_path}\n", flush=True)
    release(model)
    RESULTS[name] = result
    return result

for _name in RUN_EXPERIMENTS:
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    try:
        run_experiment(_name)
    except Exception as _err:
        import traceback
        traceback.print_exc()
        print(f"[{_name}] FAILED: {_err} - continuing with the next experiment.\n")
        release()

Loaded Phase-1 model D (GPT decoder): 8/8 layers, 64.57M parameters
[S2_D_lorma] parameters: 65.66M total | trained: 0.786M of the 65.36M language-model weights + 0.301M head (1.66% of all weights)
tokenised documents for model D: loaded from /kaggle/working/cs781_phase5/cache/chunks_4x512_569904026f36.pkl


S2_D_lorma epoch 1/1:   0%|          | 0/7670 [00:00<?, ?it/s]

dev evaluation:   0%|          | 0/595 [00:00<?, ?it/s]

[S2_D_lorma] epoch 1.00 | train loss 0.3388 | dev macro-F1 0.2806 (one threshold 0.47)  <- best | eval 215 s


dev prediction:   0%|          | 0/595 [00:00<?, ?it/s]

[S2_D_lorma] DEV macro-F1 0.3133 with per_label thresholds (held-out estimate: one threshold 0.2803, per statute 0.2967) | micro-F1 0.4024 | exact-set accuracy 0.0087 | statutes per case 6.16 (gold 3.92) | best at epoch 1.0


test prediction:   0%|          | 0/760 [00:00<?, ?it/s]

[S2_D_lorma] test: 13,019 cases, 6.03 statutes per case -> /kaggle/working/cs781_phase5/submissions/phase5_S2_D-GPT400M_LoRMA-qv-r32_LabelAttn_4x512_dev0.3133.zip



## 11. Results and analysis

In [11]:
finished = {name: r for name, r in RESULTS.items() if r is not None}
for _name in EXPERIMENTS:
    _path = os.path.join(WORK_ROOT, _name, "results.json")
    if _name not in finished and os.path.exists(_path):
        finished[_name] = load_json_file(_path)

rows = []
for name, r in sorted(finished.items(), key=lambda kv: -kv[1]["dev"]["macro_f1"]):
    cfg, dev = r["config"], r["dev"]
    technique = cfg["technique"] + (f"-{cfg['peft_targets']}-r{cfg['lora_r']}" if cfg["technique"] in ("lora", "lorma") else "") \
        + (" + KD" if cfg["teacher"] else "")
    rows.append({"experiment": name, "model": cfg["backbone"] + (f"[:{cfg['n_layers']}]" if cfg["n_layers"] else ""),
                 "technique": technique, "head": cfg["head"], "reads": f"{cfg['max_chunks']}x{cfg['chunk_tokens']}",
                 "dev_macro_f1": dev["macro_f1"], "held_out_f1": max(dev["held_out_macro_f1"].values()),
                 "thresholds": dev["threshold_strategy"], "dev_micro_f1": dev["micro_f1"],
                 "dev_subset_acc": dev["subset_accuracy"], "statutes_per_case": dev["predicted_per_case"],
                 "best_epoch": dev["best_epoch"], "params_M": dev["parameters"]["total"] / 1e6,
                 "trainable_M": dev["parameters"]["trainable"] / 1e6, "trainable_%": dev["parameters"]["trainable_percent"],
                 "train_min": dev["train_seconds"] / 60, "peak_mem_GB": dev["peak_memory_mb"] / 1000,
                 "test_cases_per_s": r["cases_per_second"], "submission": os.path.basename(r["submission"])})

if not rows:
    print("No finished experiments yet.")
else:
    header = (f"{'experiment':20s} {'model':6s} {'technique':16s} {'macroF1':>8s} {'heldout':>8s} {'thr':>9s} {'microF1':>8s} "
              f"{'exact':>6s} {'#/case':>6s} {'params':>7s} {'trained':>8s} {'%':>6s} {'train':>6s} {'mem':>5s} {'case/s':>7s}")
    print(f"Dev set ({len(CASES['dev']):,} cases, gold {Y['dev'].sum(1).mean():.2f} statutes per case). macroF1 = mean F1 over all "
          f"{NUM_LABELS} statutes with thresholds tuned on dev; heldout = the same when thresholds are tuned on the OTHER half of dev "
          f"(an honest estimate); exact = whole statute set correct.\n" + header + "\n" + "-" * len(header))
    for x in rows:
        print(f"{x['experiment']:20s} {x['model']:6s} {x['technique']:16s} {x['dev_macro_f1']:8.4f} {x['held_out_f1']:8.4f} "
              f"{x['thresholds']:>9s} {x['dev_micro_f1']:8.4f} {x['dev_subset_acc']:6.3f} {x['statutes_per_case']:6.2f} "
              f"{x['params_M']:6.1f}M {x['trainable_M']:7.3f}M {x['trainable_%']:6.2f} {x['train_min']:5.1f}m "
              f"{x['peak_mem_GB']:4.1f}G {x['test_cases_per_s']:7.0f}")
    with open(os.path.join(WORK_ROOT, "results_summary.csv"), "w", encoding="utf-8") as f:
        f.write(",".join(rows[0].keys()) + "\n")
        for x in rows:
            f.write(",".join(str(round(v, 5)) if isinstance(v, float) else str(v) for v in x.values()) + "\n")

    print("\nDev macro-F1 by document length:")
    buckets = [lab for _, _, lab in LENGTH_BUCKETS]
    print(f"{'experiment':20s} " + " ".join(f"{b:>16s}" for b in buckets))
    for x in rows:
        by = finished[x["experiment"]]["dev"]["by_length"]
        print(f"{x['experiment']:20s} " + " ".join(f"{by[b]['macro_f1']:8.4f} ({by[b]['cases']:5d})" if b in by else f"{'-':>16s}"
                                                    for b in buckets))
    print("\nDev F1 by statute frequency (training documents per statute, split into thirds: rare / middle / frequent):")
    _freq = Y["train"].sum(0)
    _thirds = np.array_split(np.argsort(_freq), 3)
    print(f"{'experiment':20s} " + " ".join(f"{lab:>22s}" for lab in (
        f"rare ({int(_freq[_thirds[0]].min())}-{int(_freq[_thirds[0]].max())})",
        f"middle ({int(_freq[_thirds[1]].min())}-{int(_freq[_thirds[1]].max())})",
        f"frequent ({int(_freq[_thirds[2]].min())}-{int(_freq[_thirds[2]].max())})")))
    for x in rows:
        f1 = np.asarray(finished[x["experiment"]]["dev"]["per_label_f1"])
        print(f"{x['experiment']:20s} " + " ".join(f"{f1[t].mean():22.4f}" for t in _thirds))
    best = finished[rows[0]["experiment"]]["dev"]
    f1 = np.asarray(best["per_label_f1"])
    order = np.argsort(f1)
    print(f"\n{rows[0]['experiment']}: weakest statutes (id: dev F1, dev cases) "
          + ", ".join(f"{STATUTE_IDS[k]}: {f1[k]:.2f} ({best['per_label_support'][k]})" for k in order[:8]))
    print(f"{rows[0]['experiment']}: strongest statutes " + ", ".join(f"{STATUTE_IDS[k]}: {f1[k]:.2f} ({best['per_label_support'][k]})"
                                                      for k in order[::-1][:8]))
    print(f"\nSummary table saved to {os.path.join(WORK_ROOT, 'results_summary.csv')}")

Dev set (10,181 cases, gold 3.92 statutes per case). macroF1 = mean F1 over all 100 statutes with thresholds tuned on dev; heldout = the same when thresholds are tuned on the OTHER half of dev (an honest estimate); exact = whole statute set correct.
experiment           model  technique         macroF1  heldout       thr  microF1  exact #/case  params  trained      %  train   mem  case/s
--------------------------------------------------------------------------------------------------------------------------------------------
S3_A4_kd             A[:4]  full + KD          0.3875   0.3715 per_label   0.4793  0.035   5.12   36.5M  36.518M 100.00  49.7m  1.9G     151
S1_B_adapter         B      adapter            0.3624   0.3447 per_label   0.4553  0.028   5.33   68.0M   1.900M   2.80 108.8m  3.7G      56
S2_D_lorma           D      lorma-qv-r32       0.3133   0.2967 per_label   0.4024  0.009   6.16   65.7M   1.087M   1.66  34.2m  2.4G      47
S4_C_bitfit          C      bitfit           

## 12. Submission files

In [12]:
if rows:
    print(f"Submission files in {SUBMISSION_DIR} (best dev score first):")
    for x in rows:
        print(f"  dev macro-F1 {x['dev_macro_f1']:.4f}   {x['submission']}")
    with zipfile.ZipFile(os.path.join(SUBMISSION_DIR, rows[0]["submission"])) as z:
        _p = json.loads(z.read("prediction.json"))
    print(f"\nprediction.json, first entries of {rows[0]['submission']}:\n ", _p[:3], f"... ({len(_p)} cases)")

Submission files in /kaggle/working/cs781_phase5/submissions (best dev score first):
  dev macro-F1 0.3875   phase5_S3_A-GPT220M-first4layers_KD-from-S1_LabelAttn_4x512_dev0.3875.zip
  dev macro-F1 0.3624   phase5_S1_B-GQA-RoPE_Adapter_LabelAttn_4x512_dev0.3624.zip
  dev macro-F1 0.3133   phase5_S2_D-GPT400M_LoRMA-qv-r32_LabelAttn_4x512_dev0.3133.zip
  dev macro-F1 0.2810   phase5_S4_C-EncDec_BitFit_LabelAttn_4x512_dev0.2810.zip

prediction.json, first entries of phase5_S3_A-GPT220M-first4layers_KD-from-S1_LabelAttn_4x512_dev0.3875.zip:
  [{'id': '100002997', 'pred': [5, 26, 34, 35, 37, 94]}, {'id': '100004477', 'pred': [5, 71, 81, 94]}, {'id': '1000086', 'pred': [5, 34, 35, 37, 94]}] ... (13019 cases)
